1. 文件结构验证
首先检查：
CSV 是否能完整读取；
有没有重复 header；
列名是否正确；
数值列是否被解析成字符串；
是否有空行和损坏行；
是否混合了多次运行结果

In [1]:
import pandas as pd

# 读取数据
packets = pd.read_csv("../dataset/OLD-ar002_et12_20260511_002-stage1_packets.csv")
# 4655906 packet_id
flows = pd.read_csv("../dataset/OLD-ar002_et12_20260511_002-stage1_flows.csv")

print("packet rows:", packets.shape)
print("flow rows:", len(flows))
print("packet columns:", len(packets.columns))
print("flow columns:", len(flows.columns))

print(packets.head())
print(flows.head())
print(packets.dtypes)
print(flows.dtypes)

packet rows: (4690999, 40)
flow rows: 177179
packet columns: 40
flow columns: 77
  record_type  packet_id      timestamp_us      ts_sec  ts_usec  \
0      packet          1  1778495267572457  1778495267   572457   
1      packet          2  1778495267572458  1778495267   572458   
2      packet          3  1778495267572471  1778495267   572471   
3      packet          4  1778495267572472  1778495267   572472   
4      packet          5  1778495267572488  1778495267   572488   

            flow_id  direction  ip_version          src_ip          dst_ip  \
0  1051312184880716          0           4   185.41.138.21  93.117.197.154   
1  1051316147181637          0           4  185.41.137.166    87.237.24.14   
2  1051370242672817          0           4  185.239.63.101   57.153.239.60   
3  1051375929166793          0           4   185.239.61.64  52.112.152.192   
4  1051443757981247          0           4   212.8.253.154   185.239.61.74   

   ...  tcp_cwr  tcp_window  tcp_header_len  ud

重复packet_id检查

In [2]:

# 判断组合 (flow_id, packet_id) 是否有重复
has_duplicates = packets.duplicated(subset=['flow_id', 'packet_id']).any()
print(f"过滤后，(flow_id, packet_id) 组合是否有重复: {has_duplicates}")

# 如果需要查看具体的重复组合及其出现次数
duplicate_counts = packets.groupby(['flow_id', 'packet_id']).size()
duplicates = duplicate_counts[duplicate_counts > 1]
print("重复的组合及其出现次数：")
print(duplicates)

# 或者直接查看重复的行（全部列）
duplicate_rows = packets[packets.duplicated(subset=['flow_id', 'packet_id'], keep=False)]
print("所有重复的行（按组合重复标记）：")
print(duplicate_rows)

过滤后，(flow_id, packet_id) 组合是否有重复: True
重复的组合及其出现次数：
flow_id           packet_id
0                 0            38
449766374495      0             2
676769937222      0             2
820016482814      0             2
1074797789364     0             2
                               ..
2247675474521897  0             2
2247752155148856  0             2
2248553611376625  0             2
2249123557281145  0             2
2249804920961132  0             2
Length: 4814, dtype: int64
所有重复的行（按组合重复标记）：
        record_type  packet_id      timestamp_us      ts_sec  ts_usec  \
617          packet          0  1778495267576720  1778495267   576720   
702          packet          0  1778495267577339  1778495267   577339   
747          packet          0  1778495267577633  1778495267   577633   
750          packet          0  1778495267577649  1778495267   577649   
2066         packet          0  1778495267585922  1778495267   585922   
...             ...        ...               ...         ...    

In [3]:
# 检查是否有任何重复的 packet_id
has_duplicates = packets['packet_id'].duplicated().any()
print(f"packet_id 是否有重复: {has_duplicates}")

# 如果想查看具体的重复值及其出现次数
duplicate_counts = packets['packet_id'].value_counts()
duplicates = duplicate_counts[duplicate_counts > 1]
print("重复的 packet_id 及其出现次数：")
print(duplicates)

packet_id 是否有重复: True
重复的 packet_id 及其出现次数：
packet_id
0    35093
Name: count, dtype: int64


重复flow_id检查

In [4]:
# 检查是否有任何重复的 flow_id
flows_filtered = flows[flows['flow_start_timestamp_us'] != 0]  #过滤了这个之后，就没有重复了

has_duplicates = flows_filtered['flow_id'].duplicated().any()
print(f"flow_id 是否有重复: {has_duplicates}")

# 如果想查看具体的重复值及其出现次数
duplicate_counts = flows_filtered['flow_id'].value_counts()
duplicates = duplicate_counts[duplicate_counts > 1]
print("重复的 flow_id 及其出现次数：")
print(duplicates)

flow_id 是否有重复: False
重复的 flow_id 及其出现次数：
Series([], Name: count, dtype: int64)


2. 基础合法性验证 同时验证时间戳组成：

In [5]:
import numpy as np

def count_invalid(name, mask):
    n = int(mask.fillna(True).sum())
    print(f"{name}: {n}")

count_invalid("invalid flow_id",
              packets["flow_id"].isna() | (packets["flow_id"] <= 0))

count_invalid("invalid timestamp",
              packets["timestamp_us"].isna() |
              (packets["timestamp_us"] <= 0))

count_invalid("invalid direction",
              ~packets["direction"].isin([0, 1]))

count_invalid("invalid IP version",
              ~packets["ip_version"].isin([4, 6]))

count_invalid("invalid protocol",
              ~packets["protocol"].between(0, 255))

count_invalid("invalid ts_usec",
              ~packets["ts_usec"].between(0, 999999))

count_invalid("invalid TTL",
              ~packets["ttl_or_hop_limit"].between(0, 255))

count_invalid("negative payload",
              packets["payload_len"] < 0)

count_invalid("payload larger than IP packet",
              packets["payload_len"] > packets["ip_len"])

expected_ts = (
    packets["ts_sec"].astype("int64") * 1_000_000
    + packets["ts_usec"].astype("int64")
)

mismatch = packets["timestamp_us"].astype("int64") != expected_ts

print("timestamp composition mismatches:", mismatch.sum())

# 3. 协议字段一致性验证

tcp = packets["has_tcp"] == 1

print("TCP but protocol != 6:",
      (tcp & (packets["protocol"] != 6)).sum())

print("non-TCP with TCP header length:",
      ((~tcp) & (packets["tcp_header_len"] != 0)).sum())

print("non-TCP with TCP window:",
      ((~tcp) & (packets["tcp_window"] != 0)).sum())

udp = packets["has_udp"] == 1

print("UDP but protocol != 17:",
      (udp & (packets["protocol"] != 17)).sum())

print("invalid UDP length:",
      (udp & (packets["udp_len"] < 8)).sum())

print("non-UDP with UDP length:",
      ((~udp) & (packets["udp_len"] != 0)).sum())

flags_rebuilt = (
      packets["tcp_fin"] * 0x01
    + packets["tcp_syn"] * 0x02
    + packets["tcp_rst"] * 0x04
    + packets["tcp_psh"] * 0x08
    + packets["tcp_ack_flag"] * 0x10
    + packets["tcp_urg"] * 0x20
    + packets["tcp_ece"] * 0x40
    + packets["tcp_cwr"] * 0x80
)

flag_mismatch = (
    (packets["has_tcp"] == 1)
    & (packets["tcp_flags"] != flags_rebuilt)
)

print("TCP flag mismatches:", flag_mismatch.sum())

invalid flow_id: 0
invalid timestamp: 0
invalid direction: 0
invalid IP version: 0
invalid protocol: 0
invalid ts_usec: 0
invalid TTL: 0
negative payload: 0
payload larger than IP packet: 0
timestamp composition mismatches: 0
TCP but protocol != 6: 0
non-TCP with TCP header length: 0
non-TCP with TCP window: 0
UDP but protocol != 17: 0
invalid UDP length: 0
non-UDP with UDP length: 0
TCP flag mismatches: 0


4. packet CSV 与 flow CSV 的一致性验证

In [6]:
packets = packets.sort_values(
    ["flow_id", "timestamp_us", "packet_id"]
).reset_index(drop=True)
grouped = packets.groupby("flow_id", sort=False)

packet_summary = grouped.agg(
    packet_rows=("packet_id", "size"),
    start_ts=("timestamp_us", "min"),
    end_ts=("timestamp_us", "max"),
    packet_alert_max=("packet_label", "max"),
)

packet_summary["fwd_packets_calc"] = grouped["direction"].apply(
    lambda x: int((x == 0).sum())
)

packet_summary["bwd_packets_calc"] = grouped["direction"].apply(
    lambda x: int((x == 1).sum())
)

packet_summary["fwd_payload_calc"] = grouped.apply(
    lambda x: x.loc[x["direction"] == 0, "payload_len"].sum(),
    include_groups=False,
)

packet_summary["bwd_payload_calc"] = grouped.apply(
    lambda x: x.loc[x["direction"] == 1, "payload_len"].sum(),
    include_groups=False,
)

packet_summary["duration_calc"] = (
    packet_summary["end_ts"] - packet_summary["start_ts"]
)

packet_summary = packet_summary.reset_index()

In [7]:

flows = flows[flows['flow_start_timestamp_us'] != 0]  #过滤了这个之后，就没有重复了

audit = flows.merge(
    packet_summary,
    on="flow_id",
    how="outer",
    indicator=True,
    validate="one_to_one",
)

print(audit["_merge"].value_counts())

checks = {
    "fwd packet mismatch":
        audit["total_fwd_packets"] != audit["fwd_packets_calc"],

    "bwd packet mismatch":
        audit["total_backward_packets"] != audit["bwd_packets_calc"],

    "fwd payload mismatch":
        audit["total_length_of_fwd_packets"] !=
        audit["fwd_payload_calc"],

    "bwd payload mismatch":
        audit["total_length_of_bwd_packets"] !=
        audit["bwd_payload_calc"],

    "start timestamp mismatch":
        audit["flow_start_timestamp_us"] != audit["start_ts"],

    "end timestamp mismatch":
        audit["flow_end_timestamp_us"] != audit["end_ts"],

    "duration mismatch":
        audit["flow_duration"] != audit["duration_calc"],

    "label mismatch":
        audit["label"] != audit["packet_alert_max"],
}

for name, mask in checks.items():
    print(name, int(mask.fillna(True).sum()))

_merge
both          177050
left_only          0
right_only         0
Name: count, dtype: int64
fwd packet mismatch 0
bwd packet mismatch 0
fwd payload mismatch 0
bwd payload mismatch 0
start timestamp mismatch 0
end timestamp mismatch 0
duration mismatch 0
label mismatch 0


5. 重新计算 IAT

In [8]:
packets["flow_iat_recomputed_us"] = (
    packets.groupby("flow_id")["timestamp_us"]
    .diff()
    .fillna(0)
)
packets["direction_iat_recomputed_us"] = (
    packets.groupby(["flow_id", "direction"])["timestamp_us"]
    .diff()
    .fillna(0)
)
print(
    "negative flow IAT:",
    (packets["flow_iat_recomputed_us"] < 0).sum()
)

print(
    "negative direction IAT:",
    (packets["direction_iat_recomputed_us"] < 0).sum()
)
flow_iat_match = np.isclose(
    packets["flow_iat_us"],
    packets["flow_iat_recomputed_us"],
)

dir_iat_match = np.isclose(
    packets["direction_iat_us"],
    packets["direction_iat_recomputed_us"],
)

print("flow IAT mismatch rate:", 1 - flow_iat_match.mean())
print("direction IAT mismatch rate:", 1 - dir_iat_match.mean())

packets["is_first_in_flow"] = (
    packets.groupby("flow_id").cumcount() == 0
).astype("int8")

packets["is_first_in_direction"] = (
    packets.groupby(["flow_id", "direction"]).cumcount() == 0
).astype("int8")

negative flow IAT: 0
negative direction IAT: 0
flow IAT mismatch rate: 0.0
direction IAT mismatch rate: 0.0


6. 数据分布验证
这决定你的 max_seq_len=L。
初始建议：
如果 95% flow 不超过 64 packets，使用 L=64；
如果 95% 不超过 128，使用 L=128；
第一轮实验一般不要超过 128；
Transformer 的计算量约随 𝐿**2增长。

In [9]:
flow_lengths = packets.groupby("flow_id").size()

print(flow_lengths.describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
))

for length in [16, 32, 64, 128, 256]:
    coverage = (flow_lengths <= length).mean()
    print(f"L={length}, complete-flow coverage={coverage:.4f}")

count    177050.000000
mean         26.171460
std         485.993344
min           1.000000
50%           2.000000
75%           7.000000
90%          14.000000
95%          29.000000
99%         308.000000
max       98614.000000
dtype: float64
L=16, complete-flow coverage=0.9150
L=32, complete-flow coverage=0.9539
L=64, complete-flow coverage=0.9722
L=128, complete-flow coverage=0.9815
L=256, complete-flow coverage=0.9883


In [10]:
# 标签分布
print(flows["label"].value_counts())
print(flows["label"].value_counts(normalize=True))

label
0    170494
1      6556
Name: count, dtype: int64
label
0    0.962971
1    0.037029
Name: proportion, dtype: float64


In [11]:
# 还要按 sequence length 检查：
length_label = flows[
    ["flow_id", "label"]
].merge(
    flow_lengths.rename("sequence_length"),
    on="flow_id",
)

print(
    length_label.groupby("label")["sequence_length"]
    .describe(percentiles=[0.5, 0.9, 0.95, 0.99])
)

          count       mean         std  min  50%   90%   95%     99%      max
label                                                                        
0      170494.0  24.564014  483.734233  1.0  2.0  14.0  28.0   290.0  98614.0
1        6556.0  67.974375  539.805931  1.0  9.0  28.0  77.0  1401.1  23221.0
